**EC2 (Elastic Compute Cloud)** is AWS's virtual server service — the most fundamental compute building block in AWS. Almost every architecture involves EC2 in some form, either directly or as the engine behind managed services.

Understanding EC2 deeply is essential for the Solutions Architect exam: instance types, purchasing options, storage, networking, and security groups are all heavily tested.

## What is EC2?

An EC2 **instance** is a virtual machine running on AWS hardware. You choose the operating system, CPU, memory, storage, and networking. AWS handles the physical hardware underneath.

### What you configure at launch

| Setting | Description |
|---|---|
| **AMI** | The OS image (Amazon Linux, Ubuntu, Windows, custom) |
| **Instance type** | CPU + memory combination (e.g. `t3.medium`) |
| **Key pair** | SSH key for Linux or RDP decryption for Windows |
| **Network** | VPC, subnet, and whether to assign a public IP |
| **Security group** | Firewall rules for inbound/outbound traffic |
| **Storage** | Root volume + optional additional EBS volumes |
| **IAM role** | Permissions the instance gets via instance profile |
| **User data** | Shell script that runs once on first boot |

## Amazon Machine Images (AMIs)

An **AMI** is a template that contains the OS, application server, and any pre-installed software for an instance.

### AMI types

| Type | Description |
|---|---|
| **AWS-provided** | Amazon Linux 2, Amazon Linux 2023, Windows Server, etc. |
| **AWS Marketplace** | Third-party AMIs — pre-configured software stacks, often with licensing included |
| **Community AMIs** | Public AMIs shared by other AWS users |
| **Custom AMIs** | AMIs you build yourself — bake your app + config into an image for fast, consistent launches |

### AMIs are region-specific
An AMI exists in a specific region. To use it in another region, you must copy it. AMI IDs differ between regions even for the same image.

### Why custom AMIs matter for architects
Instead of running a long User Data script on every launch, bake everything into an AMI. New instances start faster, and the configuration is identical every time — critical for Auto Scaling groups.

## Instance Types

Instance types follow the naming pattern: `family` + `generation` + `size` — e.g. `m7g.xlarge`.

### Instance families

| Family | Optimised for | Examples | Use cases |
|---|---|---|---|
| **General Purpose** | Balanced CPU/memory/network | `t3`, `m7g`, `m6i` | Web servers, dev environments, small DBs |
| **Compute Optimised** | High CPU ratio | `c7g`, `c6i`, `c5` | Batch processing, HPC, gaming servers, ML inference |
| **Memory Optimised** | High RAM ratio | `r7g`, `r6i`, `x2idn` | In-memory DBs, SAP HANA, large caches |
| **Storage Optimised** | High disk throughput/IOPS | `i4i`, `d3`, `h1` | NoSQL DBs, data warehouses, distributed file systems |
| **Accelerated Computing** | GPUs / custom chips | `p4`, `g5`, `trn1`, `inf2` | ML training/inference, video rendering, HPC |
| **HPC Optimised** | Tightly coupled HPC | `hpc7g` | Large-scale simulations, CFD |

### T-series burstable instances
The `t3` / `t4g` family runs at a **baseline CPU** and earns CPU credits when idle. Credits are spent when CPU bursts above baseline. Ideal for workloads with occasional spikes (dev/test, low-traffic web servers). `T3 Unlimited` mode allows sustained bursting at extra cost.

## EC2 Purchasing Options

This is one of the most exam-heavy topics in EC2. Know when to use each.

| Option | Commitment | Discount vs On-Demand | Best for |
|---|---|---|---|
| **On-Demand** | None | 0% | Short-term, unpredictable workloads |
| **Reserved Instances (1yr)** | 1 year | ~40% | Steady-state workloads you'll run 24/7 |
| **Reserved Instances (3yr)** | 3 years | ~60–72% | Long-running stable workloads |
| **Savings Plans** | 1 or 3 years | ~66% | Flexible — applies to EC2, Lambda, Fargate |
| **Spot Instances** | None | ~90% | Fault-tolerant, flexible workloads (can be interrupted) |
| **Dedicated Hosts** | On-demand or reserved | — | Bring-your-own-license (BYOL), compliance |
| **Dedicated Instances** | On-demand | Small premium | Compliance needing physical isolation, no BYOL |
| **Capacity Reservations** | None | 0% | Reserve capacity in a specific AZ, charged whether used or not |

### Reserved Instances — payment options
- **All Upfront** — pay everything now, maximum discount
- **Partial Upfront** — pay some now, rest monthly
- **No Upfront** — pay monthly, smallest discount

### Reserved Instances — scope
- **Regional** — applies to any AZ in the region; can change instance size within the family
- **Zonal** — reserved capacity in a specific AZ; capacity reservation guaranteed

### Spot Instances
Spot instances use spare AWS capacity. AWS can **reclaim** them with a 2-minute warning when capacity is needed. Use them for:
- Batch jobs, data processing, CI/CD pipelines
- Stateless web workers behind a load balancer
- Big data (EMR, Spark)

**Never use Spot for:** databases, anything that cannot tolerate interruption, or jobs that cannot checkpoint state.

## Security Groups

A **security group** is a virtual firewall for EC2 instances, controlling inbound and outbound traffic at the instance level.

### Key rules

| Property | Detail |
|---|---|
| **Stateful** | If inbound traffic is allowed, the response is automatically allowed outbound (and vice versa) |
| **Allow rules only** | You can only add Allow rules — there is no explicit Deny |
| **Default inbound** | All traffic blocked unless a rule allows it |
| **Default outbound** | All traffic allowed by default |
| **Multiple SGs** | An instance can have multiple security groups; rules are unioned |
| **Source** | Can be a CIDR block, a single IP, or another security group ID |

### Referencing security groups
Instead of hardcoding IP ranges, you can set the source of an inbound rule to another security group. This is the recommended pattern — e.g. allow the app tier's security group to talk to the DB tier's security group.

```
DB Security Group
  Inbound: port 5432  source: app-sg-id   ← only instances in app-sg can connect
```

> Security groups are **not** a substitute for network ACLs (NACLs). SGs operate at the instance level; NACLs operate at the subnet level and support Deny rules.

## EC2 Storage Options

| Storage | Type | Persists after stop/terminate? | Use case |
|---|---|---|---|
| **EBS** | Network-attached block storage | Yes (stop); configurable (terminate) | Root volume, databases, persistent data |
| **Instance Store** | Physically attached to the host | No — lost on stop/terminate/failure | Temporary buffers, caches, scratch space |
| **EFS** | Managed NFS (network file system) | Yes | Shared storage across multiple instances |
| **S3** | Object storage | Yes | Static assets, backups, data lake |

### EBS volume types (brief overview)

| Type | Class | Best for |
|---|---|---|
| `gp3` | General purpose SSD | Most workloads — default choice |
| `gp2` | General purpose SSD | Legacy; gp3 is better and cheaper |
| `io2 Block Express` | Provisioned IOPS SSD | High-performance DBs, sub-millisecond latency |
| `st1` | Throughput HDD | Big data, log processing, sequential reads |
| `sc1` | Cold HDD | Infrequently accessed data, lowest cost |

## EC2 Networking

### Public vs Private IP

| | Public IP | Private IP | Elastic IP |
|---|---|---|---|
| **Assigned** | Automatically at launch (if enabled) | Always assigned | Manually allocated |
| **Persists on stop** | No — changes on restart | Yes | Yes |
| **Reachable from** | Internet | Within VPC | Internet |
| **Cost** | Free while running | Free | Free while attached to running instance; charged when unattached |

### Elastic IP
An **Elastic IP (EIP)** is a static public IPv4 address. Use it when you need a fixed IP — e.g. whitelisting, DNS A records pointing to a single instance. Limit: 5 per region by default.

### Enhanced Networking
For high-throughput, low-latency workloads, use **Enhanced Networking** (SR-IOV). Most modern instance types support it automatically. It delivers higher packets-per-second and lower latency than traditional virtualised networking.

## User Data & Instance Metadata

### User Data
A shell script (or cloud-init directive) passed at launch that runs **once** as root on first boot. Used to:
- Install packages
- Pull application code
- Configure the OS
- Register with a service

```bash
#!/bin/bash
yum update -y
yum install -y httpd
systemctl start httpd
systemctl enable httpd
echo "<h1>Hello from EC2</h1>" > /var/www/html/index.html
```

### Instance Metadata Service (IMDS)
Running code on an instance can query the metadata service to discover information about itself — instance ID, AZ, public IP, IAM role credentials, etc.

- **IMDSv1** — simple GET request, no authentication
- **IMDSv2** — requires a session token first (more secure; recommended)

```bash
# IMDSv2 — get instance ID
TOKEN=$(curl -s -X PUT "http://169.254.169.254/latest/api/token" \
  -H "X-aws-ec2-metadata-token-ttl-seconds: 21600")
curl -s -H "X-aws-ec2-metadata-token: $TOKEN" \
  http://169.254.169.254/latest/meta-data/instance-id
```

## Working with EC2 via boto3

In [ ]:
import boto3

ec2 = boto3.client('ec2', region_name='us-east-1')
ec2_resource = boto3.resource('ec2', region_name='us-east-1')

In [ ]:
# List all running instances with their type and AZ
response = ec2.describe_instances(
    Filters=[{'Name': 'instance-state-name', 'Values': ['running']}]
)
for reservation in response['Reservations']:
    for instance in reservation['Instances']:
        name = next(
            (t['Value'] for t in instance.get('Tags', []) if t['Key'] == 'Name'), '-'
        )
        print(f"{instance['InstanceId']}  {instance['InstanceType']:<15}  "
              f"{instance['Placement']['AvailabilityZone']}  {name}")

In [ ]:
# Launch a t3.micro with a simple web server user data script
user_data = """#!/bin/bash
yum update -y
yum install -y httpd
systemctl start httpd
systemctl enable httpd
echo "<h1>Hello from EC2</h1>" > /var/www/html/index.html
"""

response = ec2.run_instances(
    ImageId='ami-0c02fb55956c7d316',      # Amazon Linux 2 in us-east-1
    InstanceType='t3.micro',
    MinCount=1,
    MaxCount=1,
    KeyName='my-key-pair',                 # replace with your key pair name
    SecurityGroupIds=['sg-xxxxxxxxx'],     # replace with your security group
    UserData=user_data,
    TagSpecifications=[{
        'ResourceType': 'instance',
        'Tags': [{'Key': 'Name', 'Value': 'demo-web-server'}]
    }]
)

instance_id = response['Instances'][0]['InstanceId']
print(f"Launched: {instance_id}")

In [ ]:
# Describe security group rules for a given SG
sg_id = 'sg-xxxxxxxxx'  # replace with your SG id
response = ec2.describe_security_groups(GroupIds=[sg_id])
sg = response['SecurityGroups'][0]

print(f"Security Group: {sg['GroupName']}")
print("\nInbound rules:")
for rule in sg['IpPermissions']:
    port = rule.get('FromPort', 'all')
    proto = rule['IpProtocol']
    sources = [r['CidrIp'] for r in rule.get('IpRanges', [])] + \
              [r['GroupId'] for r in rule.get('UserIdGroupPairs', [])]
    print(f"  {proto:<6} port {port:<6} from {sources}")

In [ ]:
# Stop an instance (data persists on EBS)
ec2.stop_instances(InstanceIds=[instance_id])
print(f"Stopping {instance_id}")

# Terminate an instance (deletes root EBS unless DeleteOnTermination=False)
# ec2.terminate_instances(InstanceIds=[instance_id])

## Summary

| Concept | Key Takeaway |
|---|---|
| AMI | OS + software template; region-specific; custom AMIs enable fast, consistent launches |
| Instance types | Family encodes the workload (general, compute, memory, storage, GPU); t-series bursts on credits |
| On-Demand | No commitment; full price; for short-term or unpredictable workloads |
| Reserved Instances | 1 or 3 year commitment; up to 72% off; for steady-state workloads |
| Spot | Up to 90% off; can be interrupted with 2-min warning; never for critical stateful work |
| Dedicated Host | Physical server for BYOL or strict compliance |
| Security Groups | Stateful, allow-only, instance-level firewall; reference SGs instead of IPs |
| Instance Store | Temporary, high-speed local disk; lost on stop/terminate |
| EBS | Persistent, network-attached block storage; survives stop; gp3 is default choice |
| Elastic IP | Static public IP; charged when unattached |
| User Data | Bootstrap script on first boot |
| IMDSv2 | Secure way for instance code to discover its own metadata and IAM credentials |